# Stage 4 — Topic interpretation and labeling

In [1]:
# Load saved assignments and text profiles only.

from pathlib import Path
import numpy as np
import pandas as pd

ARTIFACTS = Path("artifacts")  # notebook is located in project root
cluster_profiles = pd.read_csv(ARTIFACTS / "final_cluster_profiles.csv")
document_clusters = pd.read_csv(
    ARTIFACTS / "final_document_clusters.csv", dtype={"id": str}
)

print(f"Cluster profiles: {len(cluster_profiles):,}")
print(f"Assigned publications: {len(document_clusters):,}")

display(
    cluster_profiles.nlargest(8, "n_papers")[
        ["cluster_id", "n_papers", "centroid_norm", "top_terms"]
    ]
)

Cluster profiles: 160
Assigned publications: 75,110


,cluster_id,n_papers,centroid_norm,top_terms
34,34,1222,0.698970,"translation, machine translation, machine, mt,..."
76,76,1100,0.626458,"llms, models, language, large language, langua..."
15,15,1015,0.728135,"rag, retrieval, retrieval augmented, augmented..."
140,140,957,0.688662,"speech, tts, text, text speech, synthesis, mod..."
47,47,833,0.680589,"tuning, fine tuning, fine, models, parameter, ..."
149,149,796,0.730350,"reasoning, models, reasoning models, thinking,..."
19,19,792,0.661906,"reward, rl, reinforcement, reinforcement learn..."
40,40,787,0.612857,"data, synthetic, synthetic data, training, qua..."


## 1 Inspecting 8 largest final clusters

In [2]:
rng = np.random.default_rng(42)
largest_eight = cluster_profiles.nlargest(8, "n_papers")

for _, profile in largest_eight.iterrows():
    c = int(profile["cluster_id"])
    members = document_clusters.loc[document_clusters["cluster_id"] == c]
    random_docs = members.sample(n=min(3, len(members)), random_state=int(rng.integers(0, 2**31 - 1)))

    print("\n" + "=" * 70)
    print(f"CLUSTER {c} — {len(members):,} papers")
    print(f"Centroid norm: {profile['centroid_norm']:.4f}")
    print("=" * 70)
    print("\nTop terms:")
    print(profile["top_terms"])
    print("\nRepresentative titles:")
    for title in str(profile["representative_titles"]).split(" | ")[:2]:
        print("-", title)
    print("\nRandom titles:")
    for title in random_docs["clean_title"]:
        print("-", title)


CLUSTER 34 — 1,222 papers
Centroid norm: 0.6990

Top terms:
translation, machine translation, machine, mt, nmt, neural machine, translations, english, quality, neural, translation quality, languages

Representative titles:
- HW-TSC's Submission to the CCMT 2024 Machine Translation Tasks
- Evaluating the Impact of Verbal Multiword Expressions on Machine Translation

Random titles:
- EBBS: An Ensemble with Bi-Level Beam Search for Zero-Shot Machine Translation
- POMP: Probability-driven Meta-graph Prompter for LLMs in Low-resource Unsupervised Neural Machine Translation
- Evaluation of Chinese-English Machine Translation of Emotion-Loaded Microblog Texts: A Human Annotated Dataset for the Quality Assessment of Emotion Translation

CLUSTER 76 — 1,100 papers
Centroid norm: 0.6265

Top terms:
llms, models, language, large language, language models, large, models llms, llm, tasks, performance, human, capabilities

Representative titles:
- Can Language Models Recognize Convincing Arguments?


## 2. Targeted inspection of ambiguous clusters

In [3]:

import pandas as pd
from pathlib import Path

docs = pd.read_csv(
    Path("../artifacts/final_document_clusters.csv")
    if Path("../artifacts/final_document_clusters.csv").exists()
    else Path("artifacts/final_document_clusters.csv")
)

cluster_col = next(
    c for c in ["cluster_id", "cluster", "cluster_label", "label"]
    if c in docs.columns
)
title_col = next(
    c for c in ["title", "paper_title", "clean_title", "title_clean"]
    if c in docs.columns
)

clusters_to_review = [76, 140, 19, 40]

for cid in clusters_to_review:
    subset = docs.loc[docs[cluster_col] == cid]

    sample = subset.sample(
        n=min(8, len(subset)),
        random_state=2026
    )

    print(f"\n{'=' * 70}")
    print(f"CLUSTER {cid} — additional random inspection")
    print(f"{'=' * 70}")

    for i, title in enumerate(sample[title_col], 1):
        print(f"{i:2d}. {title}")



CLUSTER 76 — additional random inspection
 1. Distilling LLMs' Decomposition Abilities into Compact Language Models
 2. Can LLMs Compute with Reasons?
 3. Analyzing Large language models chatbots: An experimental approach using a probability test
 4. Stereotype Detection in LLMs: A Multiclass, Explainable, and Benchmark-Driven Approach
 5. The Trilemma of Truth in Large Language Models
 6. Large Language Models Understand Layout
 7. Continual Learning for Large Language Models: A Survey
 8. An Analysis and Mitigation of the Reversal Curse

CLUSTER 140 — additional random inspection
 1. Finding A Voice: Exploring the Potential of African American Dialect and Voice Generation for Chatbots
 2. A Study of Modeling Rising Intonation in Cantonese Neural Speech Synthesis
 3. Application of Knowledge Distillation to Multi-task Speech Representation Learning
 4. Does human speech follow Benford's Law?
 5. Self-Supervised Models of Speech Infer Universal Articulatory Kinematics
 6. Enhancing Sp

## 3. Cluster interpretation, inittial approach

In [4]:
from pathlib import Path
import pandas as pd

# Load existing final artifacts
paths = [Path("artifacts"), Path("../artifacts")]
base = next(
    p for p in paths
    if (p / "final_document_clusters.csv").exists()
)

docs = pd.read_csv(base / "final_document_clusters.csv")
profiles = pd.read_csv(base / "final_cluster_profiles.csv")

def identify_column(df, names):
    return next((c for c in names if c in df.columns), None)

doc_cluster = identify_column(
    docs, ["cluster_id", "cluster", "cluster_label", "label"]
)
profile_cluster = identify_column(
    profiles, ["cluster_id", "cluster", "cluster_label", "label"]
)
title_col = identify_column(
    docs, ["title", "paper_title", "clean_title", "title_clean"]
)

assert doc_cluster and profile_cluster and title_col

# Clusters already reviewed
reviewed = {34, 76, 15, 140, 47, 149, 19, 40}

# Select the next ten largest unreviewed clusters
sizes = docs[doc_cluster].value_counts().sort_values(ascending=False)
batch_ids = [int(cid) for cid in sizes.index if int(cid) not in reviewed][:10]

# Find vocabulary and representative-example columns
vocab_cols = [
    c for c in profiles.columns
    if any(s in c.lower() for s in ("term", "keyword", "vocab"))
]
example_cols = [
    c for c in profiles.columns
    if any(s in c.lower() for s in ("represent", "example"))
]

print("Selected clusters:", batch_ids)
print("Vocabulary fields:", vocab_cols)
print("Representative fields:", example_cols)

for cid in batch_ids:
    subset = docs.loc[docs[doc_cluster] == cid]
    profile = profiles.loc[profiles[profile_cluster] == cid]

    print("\n" + "=" * 70)
    print(f"CLUSTER {cid} — {len(subset):,} papers")
    print("=" * 70)

    if len(profile) == 1:
        for col in vocab_cols + example_cols:
            value = profile.iloc[0][col]
            print(f"{col}: {str(value)[:800]}")
    else:
        print("Profile unavailable or not unique.")

    print("\nRANDOM TITLES:")
    sample = subset.sample(
        n=min(4, len(subset)),
        random_state=2026
    )

    for i, title in enumerate(sample[title_col], 1):
        print(f"{i}. {title}")


Selected clusters: [21, 57, 22, 96, 99, 60, 73, 43, 71, 101]
Vocabulary fields: ['top_terms']
Representative fields: ['representative_titles']

CLUSTER 21 — 763 papers
top_terms: knowledge, commonsense, llms, models, language, external, language models, external knowledge, factual, commonsense knowledge, reasoning, model
representative_titles: Knowledge Card: Filling LLMs' Knowledge Gaps with Plug-in Specialized Language Models | InfuserKI: Enhancing Large Language Models with Knowledge Graphs via Infuser-Guided Knowledge Integration | Memorizing is Not Enough: Deep Knowledge Injection Through Reasoning

RANDOM TITLES:
1. Large Scale Knowledge Washing
2. DisentQA: Disentangling Parametric and Contextual Knowledge with Counterfactual Question Answering
3. To Trust or Not to Trust? Enhancing Large Language Models' Situated Faithfulness to External Contexts
4. Knowledge Rumination for Pre-trained Language Models

CLUSTER 57 — 762 papers
top_terms: social, media, social media, posts, conte

In [5]:
# Let's use cosine distance and average-linkage hierarchical clustering to organize the 160 existing cluster centers.

from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import normalize
from scipy.cluster.hierarchy import linkage, fcluster

base = next(p for p in [Path("artifacts"), Path("../artifacts")]
            if (p / "final_kmeans_160.joblib").exists())

km = joblib.load(base / "final_kmeans_160.joblib")
profiles = pd.read_csv(base / "final_cluster_profiles.csv")
id_col = next(c for c in ["cluster_id", "cluster", "label"]
              if c in profiles.columns)
profiles = profiles.set_index(id_col)

# Hierarchical grouping of the existing K-Means centroids
Z = linkage(normalize(km.cluster_centers_),
            method="average", metric="cosine")

groups = fcluster(Z, t=20, criterion="maxclust")
sizes = np.bincount(km.labels_, minlength=160)

# Display the largest constituent clusters of each group
for g in sorted(set(groups),
                key=lambda x: -sizes[groups == x].sum()):
    ids = np.where(groups == g)[0]
    largest = sorted(ids, key=lambda i: -sizes[i])[:5]
    descriptors = [
        f"{i}: {str(profiles.loc[i, 'top_terms']).split(',')[0]}"
        for i in largest
    ]
    print(f"Group {g:2d} | {sizes[ids].sum():5d} papers "
          f"| {len(ids):2d} clusters | " + "; ".join(descriptors))


Group 10 | 59765 papers | 120 clusters | 34: translation; 76: llms; 15: rag; 47: tuning; 149: reasoning
Group  7 |  4247 papers |  8 clusters | 140: speech; 22: asr; 72: speech; 58: emotion; 127: end
Group  2 |  2834 papers |  7 clusters | 57: social; 95: detection; 91: hate; 31: news; 98: news
Group  4 |  1356 papers |  5 clusters | 69: memory; 45: pruning; 74: kv; 80: quantization; 118: compression
Group  6 |  1072 papers |  2 clusters | 43: bias; 93: gender
Group  9 |   789 papers |  2 clusters | 42: event; 136: temporal
Group  8 |   674 papers |  2 clusters | 115: decoding; 106: diffusion
Group  1 |   565 papers |  2 clusters | 65: privacy; 107: unlearning
Group 17 |   547 papers |  1 clusters | 36: legal
Group  3 |   385 papers |  1 clusters | 48: topic
Group 16 |   364 papers |  1 clusters | 10: fact
Group 14 |   351 papers |  1 clusters | 17: arabic
Group 19 |   332 papers |  1 clusters | 2: sql
Group 12 |   306 papers |  1 clusters | 89: causal
Group  5 |   299 papers |  1 clus

## 4. Finalise the automated topic clustering

In [6]:

from pathlib import Path
import pandas as pd

base = next(p for p in [Path("artifacts"), Path("../artifacts")]
            if (p / "final_cluster_profiles.csv").exists())

docs = pd.read_csv(base / "final_document_clusters.csv")
profiles = pd.read_csv(base / "final_cluster_profiles.csv")

summary = profiles[["cluster_id", "top_terms"]].copy()
summary["papers"] = summary["cluster_id"].map(
    docs["cluster_id"].value_counts()
)
summary["share_pct"] = 100 * summary["papers"] / len(docs)
summary["descriptor"] = summary["top_terms"].str.split(",").apply(
    lambda terms: " / ".join(t.strip() for t in terms[:3])
)

summary = summary.sort_values("papers", ascending=False)
columns = ["cluster_id", "papers", "share_pct", "descriptor"]

summary[columns].to_csv(base / "automatic_topic_summary.csv", index=False)

print("Top 20 coverage:",
      f"{summary.head(20)['share_pct'].sum():.1f}%")
display(summary[columns].head(20).round(2))


Top 20 coverage: 21.7%


,cluster_id,papers,share_pct,descriptor
34,34,1222,1.63,translation / machine translation / machine
76,76,1100,1.46,llms / models / language
15,15,1015,1.35,rag / retrieval / retrieval augmented
140,140,957,1.27,speech / tts / text
47,47,833,1.11,tuning / fine tuning / fine
149,149,796,1.06,reasoning / models / reasoning models
19,19,792,1.05,reward / rl / reinforcement
40,40,787,1.05,data / synthetic / synthetic data
21,21,763,1.02,knowledge / commonsense / llms
22,22,762,1.01,asr / speech / speech recognition
